# Transcriptomics Preprocessing for Oligometastatic versus Polymetastatic Patients

This notebook preprocesses transcriptomics TPM data for downstream modelling of oligometastatic versus polymetastatic breast cancer.

Inputs:

- `rnaseq_tpm_norm.txt` contains oligometastatic patients;
- `TPM_prim_polymets_ALL.tsv` contains polymetastatic patients.

Therefore, the target label is assigned according to the source file:

- `Polymetastatic = 0` for samples from `rnaseq_tpm_norm.txt`;
- `Polymetastatic = 1` for samples from `TPM_prim_polymets_ALL.tsv`.

Both files contain genes as rows and samples as columns. The preprocessing workflow converts both matrices to sample-by-gene format and concatenates them by rows.

The transcriptomics source file is retained only as metadata and is not used as a model feature. However, because each metastatic group comes from a different transcriptomics file, the class label and source file are aligned. Therefore, downstream models may capture technical differences between files, processing pipelines, or batches in addition to biological differences between oligometastatic and polymetastatic patients.
For this reason, modelling results should be interpreted cautiously.

In [1]:
# Display precision only; analytical values and saved artifacts stay unchanged.
from pathlib import Path as _DisplayPath
import sys as _display_sys
_display_root = _DisplayPath.cwd()
while not (_display_root / "scripts").is_dir() and _display_root != _display_root.parent:
    _display_root = _display_root.parent
if str(_display_root) not in _display_sys.path:
    _display_sys.path.insert(0, str(_display_root))
from scripts.OligoPolyThesis.utils_ml_oligopoly import thesis_display, thesis_print
from scripts.OligoPolyThesis.utils_ml_oligopoly import (
    normalise_transcriptomics_sample_id,
    check_duplicate_genes,
    summarise_expression_values,
    convert_to_sample_by_gene_matrix,
)

import os
import numpy as np
import pandas as pd

## Define Input and Output Paths

We define the paths to the two transcriptomics input files and the output directory where preprocessed datasets and summary files will be saved.

In [2]:
transcriptomics_dir = "/mnt/data3/MCPP-Breast/Transcriptomic-Data"

oligo_file_path = os.path.join(
    transcriptomics_dir,
    "rnaseq_tpm_norm.txt"
)

poly_file_path = os.path.join(
    transcriptomics_dir,
    "TPM_prim_polymets_ALL.tsv"
)

output_dir = os.path.join(
    transcriptomics_dir,
    "Preprocessed"
)


os.makedirs(output_dir, exist_ok=True)

thesis_print("Oligometastatic transcriptomics file:")
thesis_print(oligo_file_path)

thesis_print("\nPolymetastatic transcriptomics file:")
thesis_print(poly_file_path)

thesis_print("\nOutput directory:")
thesis_print(output_dir)

Oligometastatic transcriptomics file:
/mnt/data3/MCPP-Breast/Transcriptomic-Data/rnaseq_tpm_norm.txt

Polymetastatic transcriptomics file:
/mnt/data3/MCPP-Breast/Transcriptomic-Data/TPM_prim_polymets_ALL.tsv

Output directory:
/mnt/data3/MCPP-Breast/Transcriptomic-Data/Preprocessed


## Load Transcriptomics Files

Both transcriptomics files are tab-separated matrices with genes as rows and samples as columns.

The oligometastatic file contains gene identifiers in `ensembl_gene_id` and gene symbols in `external_gene_name`.

The polymetastatic file contains gene identifiers in `gene_id` and gene symbols in `gene_name`.

In [3]:
oligo_raw_df = pd.read_csv(
    oligo_file_path,
    sep="\t"
)

poly_raw_df = pd.read_csv(
    poly_file_path,
    sep="\t"
)

thesis_print("Oligometastatic raw shape:")
thesis_print(oligo_raw_df.shape)

thesis_print("\nPolymetastatic raw shape:")
thesis_print(poly_raw_df.shape)

thesis_print("\nOligometastatic first rows:")
thesis_display(oligo_raw_df.head())

thesis_print("\nPolymetastatic first rows:")
thesis_display(poly_raw_df.head())

Oligometastatic raw shape:
(19434, 56)

Polymetastatic raw shape:
(20089, 87)

Oligometastatic first rows:


,ensembl_gene_id,external_gene_name,101B.WTS_S57,108B.WTS_S64,10B.WTS_S24,110B.WTS_S66,115B.WTS_S2,11B.WTS_S25,128B.WTS_S15,12B.WTS_S26,...,38B.WTS_S36,39B.WTS_S14,3B.WTS_S17,41B.WTS_S39,43B.WTS_S41,5B.WTS_S19,65B.WTS_S21,76B.WTS_S32,7B.WTS_S21,84B.WTS_S40
0,ENSG00000000003,TSPAN6,10.914,20.811,24.960,17.927,9.825,8.201,21.491,0.000,...,0.000,57.040,16.207,22.843,21.566,4.548,0.0,6.677,0.000,11.890
1,ENSG00000000005,TNMD,0.000,0.000,0.000,0.000,8.025,0.409,0.697,0.000,...,0.000,0.000,0.764,0.000,0.425,1.058,0.0,0.000,0.000,13.898
2,ENSG00000000419,DPM1,136.925,101.128,62.176,137.681,117.067,109.009,132.064,105.013,...,220.792,106.113,62.082,155.081,86.022,56.374,0.0,215.647,66.217,106.371
3,ENSG00000000457,SCYL3,42.470,12.279,22.519,11.299,20.952,19.334,24.650,15.042,...,13.817,0.943,27.188,16.423,22.839,12.956,0.0,13.502,0.669,45.747
4,ENSG00000000460,FIRRM,44.341,27.810,52.028,21.548,46.841,19.261,13.543,0.000,...,2.862,8.686,22.911,12.279,31.680,18.376,0.0,112.459,0.000,16.055



Polymetastatic first rows:


,gene_id,gene_name,100BC,201BC,102BC,103BC,104BC,106BC,107BC,109BC,...,70BC,73BC,83BC,85BC,88BC,8BC,93BC,99BC,9BC,202BC
0,ENSG00000000003,TSPAN6,12.470,10.353,0.000,6.620,22.565,23.083,22.196,26.098,...,4.979,7.589,16.122,29.005,0.0,13.426,5.307,5.519,16.447,14.182
1,ENSG00000000005,TNMD,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,6.560,0.0,0.000,0.000,0.000,3.221,57.452
2,ENSG00000000419,DPM1,251.748,132.432,104.333,130.163,40.414,95.576,72.779,133.279,...,37.250,74.999,8.291,88.558,0.0,69.959,106.644,267.420,87.565,14.933
3,ENSG00000000457,SCYL3,24.827,41.038,29.732,20.398,27.525,27.009,26.350,25.936,...,5.237,37.229,53.048,34.862,0.0,9.792,24.438,44.055,24.839,3.641
4,ENSG00000000460,FIRRM,0.389,42.473,13.100,22.528,20.059,27.795,10.096,37.838,...,17.970,23.365,2.210,24.102,0.0,28.249,23.778,51.596,32.732,0.000


## Define Gene Metadata and Sample Columns

We define the gene identifier columns, gene symbol columns, and sample columns for each transcriptomics file.

The Ensembl gene identifier is used as the main gene identifier because it is less ambiguous than gene symbols.

In [4]:
oligo_gene_id_col = "ensembl_gene_id"
oligo_gene_name_col = "external_gene_name"

poly_gene_id_col = "gene_id"
poly_gene_name_col = "gene_name"

oligo_sample_cols = [
    col for col in oligo_raw_df.columns
    if col not in [oligo_gene_id_col, oligo_gene_name_col]
]

poly_sample_cols = [
    col for col in poly_raw_df.columns
    if col not in [poly_gene_id_col, poly_gene_name_col]
]

thesis_print("Number of oligo samples:")
thesis_print(len(oligo_sample_cols))

thesis_print("\nNumber of poly samples:")
thesis_print(len(poly_sample_cols))

thesis_print("\nExample oligo sample columns:")
thesis_print(oligo_sample_cols[:10])

thesis_print("\nExample poly sample columns:")
thesis_print(poly_sample_cols[:10])

Number of oligo samples:
54

Number of poly samples:
85

Example oligo sample columns:
['101B.WTS_S57', '108B.WTS_S64', '10B.WTS_S24', '110B.WTS_S66', '115B.WTS_S2', '11B.WTS_S25', '128B.WTS_S15', '12B.WTS_S26', '133B.WTS_S20', '134B.WTS_S21']

Example poly sample columns:
['100BC', '201BC', '102BC', '103BC', '104BC', '106BC', '107BC', '109BC', '116BC', '123BC']


## Normalise Sample Identifiers

Sample identifiers have different formats across files.

Examples:

- `101B.WTS_S57` becomes `101B`;
- `100BC` becomes `100B`.

This normalisation creates consistent sample identifiers after concatenation.

In [5]:
oligo_sample_ids_normalised = [
    normalise_transcriptomics_sample_id(sample_id)
    for sample_id in oligo_sample_cols
]

poly_sample_ids_normalised = [
    normalise_transcriptomics_sample_id(sample_id)
    for sample_id in poly_sample_cols
]

thesis_print("Example normalised oligo sample IDs:")
thesis_print(oligo_sample_ids_normalised[:10])

thesis_print("\nExample normalised poly sample IDs:")
thesis_print(poly_sample_ids_normalised[:10])

Example normalised oligo sample IDs:
['101B', '108B', '10B', '110B', '115B', '11B', '128B', '12B', '133B', '134B']

Example normalised poly sample IDs:
['100B', '201B', '102B', '103B', '104B', '106B', '107B', '109B', '116B', '123B']


## Check Sample Overlap Between Files

The two transcriptomics files are expected to contain different patient groups.

We check whether there are duplicated or overlapping sample identifiers after normalisation.

In [6]:
oligo_sample_set = set(oligo_sample_ids_normalised)
poly_sample_set = set(poly_sample_ids_normalised)

overlapping_samples = sorted(oligo_sample_set & poly_sample_set)

sample_overlap_summary_df = pd.DataFrame({
    "metric": [
        "n_oligo_samples",
        "n_poly_samples",
        "n_overlapping_samples",
        "n_duplicated_oligo_sample_ids",
        "n_duplicated_poly_sample_ids"
    ],
    "value": [
        len(oligo_sample_ids_normalised),
        len(poly_sample_ids_normalised),
        len(overlapping_samples),
        len(oligo_sample_ids_normalised) - len(oligo_sample_set),
        len(poly_sample_ids_normalised) - len(poly_sample_set)
    ]
})

thesis_display(sample_overlap_summary_df)

thesis_print("Overlapping sample IDs:")
thesis_print(overlapping_samples)

sample_overlap_summary_df.to_csv(
    os.path.join(output_dir, "transcriptomics_sample_overlap_summary.csv"),
    index=False
)

,metric,value
0,n_oligo_samples,54
1,n_poly_samples,85
2,n_overlapping_samples,0
3,n_duplicated_oligo_sample_ids,0
4,n_duplicated_poly_sample_ids,0


Overlapping sample IDs:
[]


## Check Gene Overlap Between Files

We compare Ensembl gene identifiers between the two files.

For concatenation and downstream modelling, we retain only genes present in both transcriptomics matrices.

In [7]:
oligo_gene_ids = set(
    oligo_raw_df[oligo_gene_id_col]
    .dropna()
    .astype(str)
)

poly_gene_ids = set(
    poly_raw_df[poly_gene_id_col]
    .dropna()
    .astype(str)
)

common_gene_ids = sorted(oligo_gene_ids & poly_gene_ids)

gene_overlap_summary_df = pd.DataFrame({
    "metric": [
        "n_oligo_genes",
        "n_poly_genes",
        "n_common_genes",
        "n_genes_only_in_oligo",
        "n_genes_only_in_poly"
    ],
    "value": [
        len(oligo_gene_ids),
        len(poly_gene_ids),
        len(common_gene_ids),
        len(oligo_gene_ids - poly_gene_ids),
        len(poly_gene_ids - oligo_gene_ids)
    ]
})

thesis_display(gene_overlap_summary_df)

gene_overlap_summary_df.to_csv(
    os.path.join(output_dir, "transcriptomics_gene_overlap_summary.csv"),
    index=False
)

,metric,value
0,n_oligo_genes,19434
1,n_poly_genes,20089
2,n_common_genes,19434
3,n_genes_only_in_oligo,0
4,n_genes_only_in_poly,655


## Check Duplicated Gene Identifiers

Duplicated gene identifiers can create duplicated feature columns after transposition.

We inspect duplicated Ensembl gene IDs and gene symbols in both files before creating the modelling matrix.

In [8]:
duplicate_gene_summary_df = pd.DataFrame([
    check_duplicate_genes(
        df=oligo_raw_df,
        gene_id_col=oligo_gene_id_col,
        gene_name_col=oligo_gene_name_col,
        dataset_name="rnaseq_tpm_norm.txt"
    ),
    check_duplicate_genes(
        df=poly_raw_df,
        gene_id_col=poly_gene_id_col,
        gene_name_col=poly_gene_name_col,
        dataset_name="TPM_prim_polymets_ALL.tsv"
    )
])

thesis_display(duplicate_gene_summary_df)

duplicate_gene_summary_df.to_csv(
    os.path.join(output_dir, "transcriptomics_duplicate_gene_summary.csv"),
    index=False
)

,dataset,n_rows,n_unique_gene_ids,n_duplicated_gene_ids,n_missing_gene_ids,n_unique_gene_names,n_duplicated_gene_names,n_missing_gene_names
0,rnaseq_tpm_norm.txt,19434,19434,0,0,19434,0,0
1,TPM_prim_polymets_ALL.tsv,20089,20089,0,0,19434,654,626


## Summarise Expression Value Distributions

We compare expression distributions before concatenation because the oligometastatic and polymetastatic samples come from separate files. If expression value distributions differ substantially, downstream models may capture file-specific technical differences in addition to biological signal.

In [9]:
expression_distribution_summary_df = pd.DataFrame([
    summarise_expression_values(
        df=oligo_raw_df,
        gene_id_col=oligo_gene_id_col,
        gene_name_col=oligo_gene_name_col,
        dataset_name="rnaseq_tpm_norm.txt"
    ),
    summarise_expression_values(
        df=poly_raw_df,
        gene_id_col=poly_gene_id_col,
        gene_name_col=poly_gene_name_col,
        dataset_name="TPM_prim_polymets_ALL.tsv"
    )
])

thesis_display(expression_distribution_summary_df)

expression_distribution_summary_df.to_csv(
    os.path.join(output_dir, "transcriptomics_expression_distribution_summary.csv"),
    index=False
)

,dataset,n_values,min,q1,median,mean,q3,q95,q99,max,n_zero,zero_fraction,n_missing
0,rnaseq_tpm_norm.txt,1049436,0.0,0.0,12.249,51.166,46.768,168.766,415.955,981235.340,319279,0.304,0
1,TPM_prim_polymets_ALL.tsv,1707565,0.0,0.0,10.789,48.342,44.622,165.077,404.156,862977.400,545894,0.320,0


## Prepare Metadata for Common Genes

We create a metadata table for genes shared by both transcriptomics files.

The Ensembl gene ID is used as the feature identifier, and gene symbols are retained for interpretation.

In [10]:
oligo_gene_metadata_df = (
    oligo_raw_df[[oligo_gene_id_col, oligo_gene_name_col]]
    .rename(
        columns={
            oligo_gene_id_col: "ensembl_gene_id",
            oligo_gene_name_col: "gene_name_oligo_file"
        }
    )
    .copy()
)

poly_gene_metadata_df = (
    poly_raw_df[[poly_gene_id_col, poly_gene_name_col]]
    .rename(
        columns={
            poly_gene_id_col: "ensembl_gene_id",
            poly_gene_name_col: "gene_name_poly_file"
        }
    )
    .copy()
)

common_gene_metadata_df = (
    pd.DataFrame({"ensembl_gene_id": common_gene_ids})
    .merge(
        oligo_gene_metadata_df,
        on="ensembl_gene_id",
        how="left"
    )
    .merge(
        poly_gene_metadata_df,
        on="ensembl_gene_id",
        how="left"
    )
)

common_gene_metadata_df["gene_name"] = (
    common_gene_metadata_df["gene_name_oligo_file"]
    .fillna(common_gene_metadata_df["gene_name_poly_file"])
)

thesis_display(common_gene_metadata_df.head())

common_gene_metadata_df.to_csv(
    os.path.join(output_dir, "transcriptomics_common_gene_metadata.csv"),
    index=False
)

,ensembl_gene_id,gene_name_oligo_file,gene_name_poly_file,gene_name
0,ENSG00000000003,TSPAN6,TSPAN6,TSPAN6
1,ENSG00000000005,TNMD,TNMD,TNMD
2,ENSG00000000419,DPM1,DPM1,DPM1
3,ENSG00000000457,SCYL3,SCYL3,SCYL3
4,ENSG00000000460,FIRRM,FIRRM,FIRRM


## Convert Transcriptomics Matrices to Sample-by-Gene Format

The raw files have genes as rows and samples as columns.

For machine-learning modelling, each file is converted to:

- rows = samples;
- columns = genes;
- values = TPM expression values.

Only common Ensembl gene IDs are retained.

In [11]:
oligo_sample_by_gene_df = convert_to_sample_by_gene_matrix(
    df=oligo_raw_df,
    gene_id_col=oligo_gene_id_col,
    gene_name_col=oligo_gene_name_col,
    common_gene_ids=common_gene_ids,
    dataset_name="rnaseq_tpm_norm.txt",
    target_value=0,
    target_label="oligometastatic"
)

poly_sample_by_gene_df = convert_to_sample_by_gene_matrix(
    df=poly_raw_df,
    gene_id_col=poly_gene_id_col,
    gene_name_col=poly_gene_name_col,
    common_gene_ids=common_gene_ids,
    dataset_name="TPM_prim_polymets_ALL.tsv",
    target_value=1,
    target_label="polymetastatic"
)

rnaseq_tpm_norm.txt sample-by-gene matrix shape:
(54, 19439)

Target distribution for rnaseq_tpm_norm.txt:


Polymetastatic
0    54
Name: count, dtype: int64

TPM_prim_polymets_ALL.tsv sample-by-gene matrix shape:
(85, 19439)

Target distribution for TPM_prim_polymets_ALL.tsv:


Polymetastatic
1    85
Name: count, dtype: int64

## Concatenate Oligometastatic and Polymetastatic Samples

The two sample-by-gene matrices are concatenated by rows.

The resulting matrix contains all transcriptomics samples, common genes as features, and the target label assigned according to the project organisation document.

In [12]:
combined_transcriptomics_df = pd.concat(
    [
        oligo_sample_by_gene_df,
        poly_sample_by_gene_df
    ],
    axis=0,
    ignore_index=True
)

thesis_print("Combined transcriptomics shape:")
thesis_print(combined_transcriptomics_df.shape)

thesis_print("\nTarget distribution:")
thesis_display(combined_transcriptomics_df["Polymetastatic"].value_counts())

thesis_print("\nSamples per source:")
thesis_display(combined_transcriptomics_df["transcriptomics_source"].value_counts())

thesis_print("\nSamples per metastatic group:")
thesis_display(combined_transcriptomics_df["metastatic_group"].value_counts())

thesis_display(combined_transcriptomics_df.head())

Combined transcriptomics shape:
(139, 19439)

Target distribution:


Polymetastatic
1    85
0    54
Name: count, dtype: int64


Samples per source:


transcriptomics_source
TPM_prim_polymets_ALL.tsv    85
rnaseq_tpm_norm.txt          54
Name: count, dtype: int64


Samples per metastatic group:


metastatic_group
polymetastatic     85
oligometastatic    54
Name: count, dtype: int64

,sample_id_original,transcriptomics__ENSG00000000003,transcriptomics__ENSG00000000005,transcriptomics__ENSG00000000419,transcriptomics__ENSG00000000457,transcriptomics__ENSG00000000460,transcriptomics__ENSG00000000938,transcriptomics__ENSG00000000971,transcriptomics__ENSG00000001036,transcriptomics__ENSG00000001084,...,transcriptomics__ENSG00000291237,transcriptomics__ENSG00000291307,transcriptomics__ENSG00000291309,transcriptomics__ENSG00000291316,transcriptomics__ENSG00000291317,transcriptomics__ENSG00000293543,sample_id_normalised,transcriptomics_source,metastatic_group,Polymetastatic
0,101B.WTS_S57,10.914,0.000,136.925,42.470,44.341,0.976,44.061,75.345,44.662,...,640.249,0.0,65.740,31.726,27.653,0.0,101B,rnaseq_tpm_norm.txt,oligometastatic,0
1,108B.WTS_S64,20.811,0.000,101.128,12.279,27.810,13.561,38.499,30.774,76.027,...,113.576,0.0,52.868,1.481,16.371,0.0,108B,rnaseq_tpm_norm.txt,oligometastatic,0
2,10B.WTS_S24,24.960,0.000,62.176,22.519,52.028,6.546,42.033,32.938,33.254,...,174.583,0.0,47.821,0.000,42.819,0.0,10B,rnaseq_tpm_norm.txt,oligometastatic,0
3,110B.WTS_S66,17.927,0.000,137.681,11.299,21.548,12.373,111.469,48.562,87.159,...,423.725,0.0,61.003,0.000,11.289,0.0,110B,rnaseq_tpm_norm.txt,oligometastatic,0
4,115B.WTS_S2,9.825,8.025,117.067,20.952,46.841,19.584,152.683,48.963,43.838,...,204.174,0.0,38.168,12.514,14.589,0.0,115B,rnaseq_tpm_norm.txt,oligometastatic,0


## Check Missing Values After Concatenation

We inspect missing values in the combined transcriptomics matrix.

Since only common genes are retained, missing expression values are expected to be absent or minimal.

In [13]:
metadata_cols = [
    "sample_id_original",
    "sample_id_normalised",
    "transcriptomics_source",
    "metastatic_group",
    "Polymetastatic"
]

expression_feature_cols = [
    col for col in combined_transcriptomics_df.columns
    if col not in metadata_cols
]

missing_summary = {
    "n_samples": combined_transcriptomics_df.shape[0],
    "n_expression_features": len(expression_feature_cols),
    "n_total_missing_expression_values": (
        combined_transcriptomics_df[expression_feature_cols]
        .isna()
        .sum()
        .sum()
    ),
    "n_features_with_missing_values": (
        combined_transcriptomics_df[expression_feature_cols]
        .isna()
        .sum()
        .gt(0)
        .sum()
    ),
    "n_samples_with_missing_values": (
        combined_transcriptomics_df[expression_feature_cols]
        .isna()
        .sum(axis=1)
        .gt(0)
        .sum()
    )
}

missing_summary_df = pd.DataFrame(
    missing_summary.items(),
    columns=["metric", "value"]
)

thesis_display(missing_summary_df)

missing_summary_df.to_csv(
    os.path.join(output_dir, "transcriptomics_missing_value_summary.csv"),
    index=False
)

,metric,value
0,n_samples,139
1,n_expression_features,19434
2,n_total_missing_expression_values,0
3,n_features_with_missing_values,0
4,n_samples_with_missing_values,0


## Apply Log2 Transformation

TPM values are right-skewed. We apply a `log2(TPM + 1)` transformation to reduce skewness and make expression values more suitable for modelling.

The target label and sample metadata are not transformed.

In [14]:
combined_transcriptomics_log2_df = combined_transcriptomics_df.copy()

combined_transcriptomics_log2_df[expression_feature_cols] = np.log2(
    combined_transcriptomics_log2_df[expression_feature_cols].astype(float) + 1
)

thesis_print("Combined log2-transformed transcriptomics shape:")
thesis_print(combined_transcriptomics_log2_df.shape)

thesis_display(combined_transcriptomics_log2_df.head())

Combined log2-transformed transcriptomics shape:
(139, 19439)


,sample_id_original,transcriptomics__ENSG00000000003,transcriptomics__ENSG00000000005,transcriptomics__ENSG00000000419,transcriptomics__ENSG00000000457,transcriptomics__ENSG00000000460,transcriptomics__ENSG00000000938,transcriptomics__ENSG00000000971,transcriptomics__ENSG00000001036,transcriptomics__ENSG00000001084,...,transcriptomics__ENSG00000291237,transcriptomics__ENSG00000291307,transcriptomics__ENSG00000291309,transcriptomics__ENSG00000291316,transcriptomics__ENSG00000291317,transcriptomics__ENSG00000293543,sample_id_normalised,transcriptomics_source,metastatic_group,Polymetastatic
0,101B.WTS_S57,3.575,0.000,7.108,5.442,5.503,0.983,5.494,6.254,5.513,...,9.325,0.0,6.060,5.032,4.841,0.0,101B,rnaseq_tpm_norm.txt,oligometastatic,0
1,108B.WTS_S64,4.447,0.000,6.674,3.731,4.848,3.864,5.304,4.990,6.267,...,6.840,0.0,5.751,1.311,4.119,0.0,108B,rnaseq_tpm_norm.txt,oligometastatic,0
2,10B.WTS_S24,4.698,0.000,5.981,4.556,5.729,2.916,5.427,5.085,5.098,...,7.456,0.0,5.609,0.000,5.453,0.0,10B,rnaseq_tpm_norm.txt,oligometastatic,0
3,110B.WTS_S66,4.242,0.000,7.116,3.620,4.495,3.741,6.813,5.631,6.462,...,8.730,0.0,5.954,0.000,3.619,0.0,110B,rnaseq_tpm_norm.txt,oligometastatic,0
4,115B.WTS_S2,3.436,3.174,6.883,4.456,5.580,4.363,7.264,5.643,5.487,...,7.681,0.0,5.292,3.756,3.962,0.0,115B,rnaseq_tpm_norm.txt,oligometastatic,0


## Filter Low-Expression Genes

Genes with very low or absent expression in most samples are unlikely to be informative and can increase overfitting in downstream models.

We retain genes expressed above a minimum TPM threshold in at least a minimum number of samples.

This filtering step is unsupervised because it does not use the target label.

In [15]:
MIN_TPM = 1
MIN_SAMPLES_EXPRESSED = 5

# Use original TPM values, not log2 values, to define expression prevalence.
expression_presence = (
    combined_transcriptomics_df[expression_feature_cols]
    .astype(float)
    .ge(MIN_TPM)
)

n_samples_expressed = expression_presence.sum(axis=0)

retained_expression_features = n_samples_expressed[
    n_samples_expressed >= MIN_SAMPLES_EXPRESSED
].index.tolist()

low_expression_filter_summary_df = pd.DataFrame({
    "feature": expression_feature_cols,
    "n_samples_expressed_tpm_ge_1": n_samples_expressed.values,
    "retained": [
        feature in retained_expression_features
        for feature in expression_feature_cols
    ]
})

thesis_print("Minimum TPM threshold:")
thesis_print(MIN_TPM)

thesis_print("\nMinimum number of samples expressed:")
thesis_print(MIN_SAMPLES_EXPRESSED)

thesis_print("\nNumber of expression features before filtering:")
thesis_print(len(expression_feature_cols))

thesis_print("\nNumber of expression features after filtering:")
thesis_print(len(retained_expression_features))

thesis_display(low_expression_filter_summary_df.head())

low_expression_filter_summary_df.to_csv(
    os.path.join(output_dir, "transcriptomics_low_expression_filter_summary.csv"),
    index=False
)

Minimum TPM threshold:
1

Minimum number of samples expressed:
5

Number of expression features before filtering:
19434

Number of expression features after filtering:
18260


,feature,n_samples_expressed_tpm_ge_1,retained
0,transcriptomics__ENSG00000000003,118,True
1,transcriptomics__ENSG00000000005,32,True
2,transcriptomics__ENSG00000000419,132,True
3,transcriptomics__ENSG00000000457,122,True
4,transcriptomics__ENSG00000000460,122,True


## Create and Inspect Final Modelling Matrix

We create the final preprocessed transcriptomics modelling matrix.

The final matrix contains:

- sample metadata;
- the target variable `Polymetastatic`;
- log2-transformed expression features retained after low-expression filtering.

The metadata column `transcriptomics_source` is retained for auditing and quality control, but it should not be used as a model feature.

Before saving, we display the final matrix and inspect its dimensions, target distribution, metadata columns, and expression feature columns.

In [16]:
final_transcriptomics_df = combined_transcriptomics_log2_df[
    metadata_cols + retained_expression_features
].copy()

thesis_print("Final transcriptomics modelling matrix shape:")
thesis_print(final_transcriptomics_df.shape)

thesis_print("\nTarget distribution:")
thesis_display(final_transcriptomics_df["Polymetastatic"].value_counts())

thesis_print("\nSamples per transcriptomics source:")
thesis_display(final_transcriptomics_df["transcriptomics_source"].value_counts())

thesis_print("\nSamples per metastatic group:")
thesis_display(final_transcriptomics_df["metastatic_group"].value_counts())

thesis_print("\nNumber of retained transcriptomics features:")
thesis_print(len(retained_expression_features))

thesis_print("\nMetadata columns:")
thesis_print(metadata_cols)

thesis_print("\nFirst 10 expression feature columns:")
thesis_print(retained_expression_features[:10])

thesis_print("\nFinal transcriptomics matrix preview:")
thesis_display(final_transcriptomics_df.head())

thesis_print("\nFinal transcriptomics matrix preview with metadata and first 10 features:")
thesis_display(
    final_transcriptomics_df[
        metadata_cols + retained_expression_features[:10]
    ].head()
)

Final transcriptomics modelling matrix shape:
(139, 18265)

Target distribution:


Polymetastatic
1    85
0    54
Name: count, dtype: int64


Samples per transcriptomics source:


transcriptomics_source
TPM_prim_polymets_ALL.tsv    85
rnaseq_tpm_norm.txt          54
Name: count, dtype: int64


Samples per metastatic group:


metastatic_group
polymetastatic     85
oligometastatic    54
Name: count, dtype: int64


Number of retained transcriptomics features:
18260

Metadata columns:
['sample_id_original', 'sample_id_normalised', 'transcriptomics_source', 'metastatic_group', 'Polymetastatic']

First 10 expression feature columns:
['transcriptomics__ENSG00000000003', 'transcriptomics__ENSG00000000005', 'transcriptomics__ENSG00000000419', 'transcriptomics__ENSG00000000457', 'transcriptomics__ENSG00000000460', 'transcriptomics__ENSG00000000938', 'transcriptomics__ENSG00000000971', 'transcriptomics__ENSG00000001036', 'transcriptomics__ENSG00000001084', 'transcriptomics__ENSG00000001167']

Final transcriptomics matrix preview:


,sample_id_original,sample_id_normalised,transcriptomics_source,metastatic_group,Polymetastatic,transcriptomics__ENSG00000000003,transcriptomics__ENSG00000000005,transcriptomics__ENSG00000000419,transcriptomics__ENSG00000000457,transcriptomics__ENSG00000000460,...,transcriptomics__ENSG00000289721,transcriptomics__ENSG00000289746,transcriptomics__ENSG00000289762,transcriptomics__ENSG00000290292,transcriptomics__ENSG00000291237,transcriptomics__ENSG00000291307,transcriptomics__ENSG00000291309,transcriptomics__ENSG00000291316,transcriptomics__ENSG00000291317,transcriptomics__ENSG00000293543
0,101B.WTS_S57,101B,rnaseq_tpm_norm.txt,oligometastatic,0,3.575,0.000,7.108,5.442,5.503,...,0.000,0.000,0.000,3.874,9.325,0.0,6.060,5.032,4.841,0.0
1,108B.WTS_S64,108B,rnaseq_tpm_norm.txt,oligometastatic,0,4.447,0.000,6.674,3.731,4.848,...,0.977,2.580,4.518,4.208,6.840,0.0,5.751,1.311,4.119,0.0
2,10B.WTS_S24,10B,rnaseq_tpm_norm.txt,oligometastatic,0,4.698,0.000,5.981,4.556,5.729,...,2.668,1.598,0.000,5.191,7.456,0.0,5.609,0.000,5.453,0.0
3,110B.WTS_S66,110B,rnaseq_tpm_norm.txt,oligometastatic,0,4.242,0.000,7.116,3.620,4.495,...,0.000,3.426,0.000,4.180,8.730,0.0,5.954,0.000,3.619,0.0
4,115B.WTS_S2,115B,rnaseq_tpm_norm.txt,oligometastatic,0,3.436,3.174,6.883,4.456,5.580,...,3.117,4.866,0.000,4.905,7.681,0.0,5.292,3.756,3.962,0.0



Final transcriptomics matrix preview with metadata and first 10 features:


,sample_id_original,sample_id_normalised,transcriptomics_source,metastatic_group,Polymetastatic,transcriptomics__ENSG00000000003,transcriptomics__ENSG00000000005,transcriptomics__ENSG00000000419,transcriptomics__ENSG00000000457,transcriptomics__ENSG00000000460,transcriptomics__ENSG00000000938,transcriptomics__ENSG00000000971,transcriptomics__ENSG00000001036,transcriptomics__ENSG00000001084,transcriptomics__ENSG00000001167
0,101B.WTS_S57,101B,rnaseq_tpm_norm.txt,oligometastatic,0,3.575,0.000,7.108,5.442,5.503,0.983,5.494,6.254,5.513,5.626
1,108B.WTS_S64,108B,rnaseq_tpm_norm.txt,oligometastatic,0,4.447,0.000,6.674,3.731,4.848,3.864,5.304,4.990,6.267,4.868
2,10B.WTS_S24,10B,rnaseq_tpm_norm.txt,oligometastatic,0,4.698,0.000,5.981,4.556,5.729,2.916,5.427,5.085,5.098,6.475
3,110B.WTS_S66,110B,rnaseq_tpm_norm.txt,oligometastatic,0,4.242,0.000,7.116,3.620,4.495,3.741,6.813,5.631,6.462,3.759
4,115B.WTS_S2,115B,rnaseq_tpm_norm.txt,oligometastatic,0,3.436,3.174,6.883,4.456,5.580,4.363,7.264,5.643,5.487,5.286


## Create Feature Name Mapping

The mapping links model feature names to Ensembl gene IDs and gene symbols for later interpretation.

In [17]:
feature_name_mapping_df = pd.DataFrame({
    "model_feature_name": retained_expression_features
})

feature_name_mapping_df["ensembl_gene_id"] = (
    feature_name_mapping_df["model_feature_name"]
    .str.replace("transcriptomics__", "", regex=False)
)

feature_name_mapping_df = feature_name_mapping_df.merge(
    common_gene_metadata_df[
        [
            "ensembl_gene_id",
            "gene_name"
        ]
    ],
    on="ensembl_gene_id",
    how="left"
)

thesis_display(feature_name_mapping_df.head())

feature_name_mapping_df.to_csv(
    os.path.join(output_dir, "transcriptomics_feature_name_mapping.csv"),
    index=False
)

,model_feature_name,ensembl_gene_id,gene_name
0,transcriptomics__ENSG00000000003,ENSG00000000003,TSPAN6
1,transcriptomics__ENSG00000000005,ENSG00000000005,TNMD
2,transcriptomics__ENSG00000000419,ENSG00000000419,DPM1
3,transcriptomics__ENSG00000000457,ENSG00000000457,SCYL3
4,transcriptomics__ENSG00000000460,ENSG00000000460,FIRRM


## Final Sanity Check Before Saving

Before saving the final matrix, we verify that:

- the number of samples matches the expected total;
- the target contains only 0 and 1;
- there are no missing expression values;
- model features are only transcriptomics expression features;
- metadata columns are excluded from the modelling feature matrix.

In [18]:
feature_cols = [
    col for col in final_transcriptomics_df.columns
    if col.startswith("transcriptomics__")
]

X = final_transcriptomics_df[feature_cols].copy()
y = final_transcriptomics_df["Polymetastatic"].astype(int)

thesis_print("X shape:")
thesis_print(X.shape)

thesis_print("\ny shape:")
thesis_print(y.shape)

thesis_print("\nTarget distribution:")
thesis_display(y.value_counts())

thesis_print("\nFeature dtype counts:")
thesis_display(X.dtypes.value_counts())

thesis_print("\nTotal missing values in X:")
thesis_print(X.isna().sum().sum())

metadata_cols_excluded = [
    col for col in final_transcriptomics_df.columns
    if not col.startswith("transcriptomics__")
]

thesis_print("\nMetadata columns excluded from X:")
thesis_print(metadata_cols_excluded)

assert X.shape[0] == y.shape[0]
assert set(y.unique()).issubset({0, 1})
assert X.isna().sum().sum() == 0
assert "transcriptomics_source" not in X.columns
assert "metastatic_group" not in X.columns
assert "Polymetastatic" not in X.columns
assert len(feature_cols) == len(retained_expression_features)

thesis_print("\nSanity check passed: final matrix is ready to be saved and used for modelling.")

X shape:
(139, 18260)

y shape:
(139,)

Target distribution:


Polymetastatic
1    85
0    54
Name: count, dtype: int64


Feature dtype counts:


float64    18260
Name: count, dtype: int64


Total missing values in X:
0

Metadata columns excluded from X:
['sample_id_original', 'sample_id_normalised', 'transcriptomics_source', 'metastatic_group', 'Polymetastatic']

Sanity check passed: final matrix is ready to be saved and used for modelling.


## Save Preprocessed Transcriptomics Outputs

We save the final preprocessed transcriptomics matrix and associated summary files.

The main output is the final sample-by-feature matrix with log2-transformed expression values.

In [19]:
final_transcriptomics_output_path = os.path.join(
    output_dir,
    "transcriptomics_oligo_poly_log2_tpm_preprocessed.csv"
)

combined_raw_tpm_output_path = os.path.join(
    output_dir,
    "transcriptomics_oligo_poly_raw_tpm_common_genes.csv"
)

final_transcriptomics_df.to_csv(
    final_transcriptomics_output_path,
    index=False
)

combined_transcriptomics_df.to_csv(
    combined_raw_tpm_output_path,
    index=False
)

thesis_print("Saved files:")
thesis_print(final_transcriptomics_output_path)
thesis_print(combined_raw_tpm_output_path)
thesis_print(os.path.join(output_dir, "transcriptomics_feature_name_mapping.csv"))
thesis_print(os.path.join(output_dir, "transcriptomics_common_gene_metadata.csv"))
thesis_print(os.path.join(output_dir, "transcriptomics_gene_overlap_summary.csv"))
thesis_print(os.path.join(output_dir, "transcriptomics_sample_overlap_summary.csv"))
thesis_print(os.path.join(output_dir, "transcriptomics_expression_distribution_summary.csv"))
thesis_print(os.path.join(output_dir, "transcriptomics_low_expression_filter_summary.csv"))
thesis_print(os.path.join(output_dir, "transcriptomics_missing_value_summary.csv"))
thesis_print(os.path.join(output_dir, "transcriptomics_duplicate_gene_summary.csv"))

Saved files:
/mnt/data3/MCPP-Breast/Transcriptomic-Data/Preprocessed/transcriptomics_oligo_poly_log2_tpm_preprocessed.csv
/mnt/data3/MCPP-Breast/Transcriptomic-Data/Preprocessed/transcriptomics_oligo_poly_raw_tpm_common_genes.csv
/mnt/data3/MCPP-Breast/Transcriptomic-Data/Preprocessed/transcriptomics_feature_name_mapping.csv
/mnt/data3/MCPP-Breast/Transcriptomic-Data/Preprocessed/transcriptomics_common_gene_metadata.csv
/mnt/data3/MCPP-Breast/Transcriptomic-Data/Preprocessed/transcriptomics_gene_overlap_summary.csv
/mnt/data3/MCPP-Breast/Transcriptomic-Data/Preprocessed/transcriptomics_sample_overlap_summary.csv
/mnt/data3/MCPP-Breast/Transcriptomic-Data/Preprocessed/transcriptomics_expression_distribution_summary.csv
/mnt/data3/MCPP-Breast/Transcriptomic-Data/Preprocessed/transcriptomics_low_expression_filter_summary.csv
/mnt/data3/MCPP-Breast/Transcriptomic-Data/Preprocessed/transcriptomics_missing_value_summary.csv
/mnt/data3/MCPP-Breast/Transcriptomic-Data/Preprocessed/transcriptom